# S&P 500 — Daily + Monthly extraction (CIZ format)

Variant of `data_extraction_sp500.ipynb` using the CRSP **CIZ tables** (`dsf_v2`, `stocknames_v2`, `stkdistributions`, `stkdelists`).

**Why this version exists**: legacy CRSP tables (`crsp.dsf`, `crsp.dsi`, `crsp.stocknames`, ...) cap at 2024-12-31 on this WRDS subscription. CIZ tables extend through 2025+.

**Key differences vs the legacy notebook**:

- Source tables: CIZ (see mapping below).
- Trading-days source: `crsp.dsf_v2` (distinct `dlycaldt`) — `crsp.dsi` only goes to 2024.
- `crsp.dsi` is **kept** for benchmarks (VWRETD/EWRETD/SPRTRN/SPINDX/...). Rows for dates 2025+ get NULL benchmarks (graceful degradation).
- Common-stock filter: legacy `shrcd in (10, 11)` becomes `securitytype='EQTY' AND securitysubtype='COM' AND sharetype in (NS, AD, SB, UG)`.
- `SHRCD` is synthesized from `usincflg` (`Y` -> 10, `N` -> 11).
- Categorical codes that were numeric in legacy are TEXT in CIZ: `DISTCD` (was int -> now `disdetailtype`), `DLSTCD` (was int -> now `delreasontype`), `HEXCD` (was int -> now `primaryexch`).
- `ISSUNO` is aliased from `nasdissuno` (NASDAQ-specific; NULL for non-NASDAQ securities).
- All output column names are kept identical to the legacy notebook so downstream pandas code can stay the same.

**CIZ -> legacy column-name mapping (applied via SQL `AS` aliases)**:

| Source | CIZ column | Output alias |
|--------|-----------|--------------|
| dsf_v2 | dlycaldt  | date         |
| dsf_v2 | dlyprc    | prc          |
| dsf_v2 | dlyret    | ret          |
| dsf_v2 | dlyretx   | retx         |
| dsf_v2 | dlyvol    | vol          |
| dsf_v2 | dlylow    | bidlo        |
| dsf_v2 | dlyhigh   | askhi        |
| dsf_v2 | dlybid    | bid          |
| dsf_v2 | dlyask    | ask          |
| dsf_v2 | dlyopen   | openprc      |
| dsf_v2 | dlynumtrd | numtrd       |
| dsf_v2 | dlycumfacpr  | cfacpr    |
| dsf_v2 | dlycumfacshr | cfacshr   |
| dsf_v2 | primaryexch  | hexcd     |
| dsf_v2 | siccd        | hsiccd    |
| dsf_v2 | nasdissuno   | issuno    |
| dsf_v2 | cusip        | cusip_hdr |
| stocknames_v2 | issuernm        | comnam |
| stocknames_v2 | shareclass      | shrcls |
| stkdistributions | disexdt      | date     |
| stkdistributions | disdivamt    | divamt   |
| stkdistributions | disfacpr     | facpr    |
| stkdistributions | disfacshr    | facshr   |
| stkdistributions | disdetailtype| distcd   |
| stkdistributions | dispaydt     | paydt    |
| stkdistributions | disrecorddt  | rcrddt   |
| stkdelists       | delistingdt  | date     |
| stkdelists       | delreasontype| dlstcd   |
| stkdelists       | deldtprc     | dlprc    |
| stkdelists       | delret       | dlret    |

In [ ]:
import wrds
import pandas as pd
import numpy as np
from pathlib import Path

## 0) WRDS connection

In [ ]:
WRDS_USERNAME = 'aadmberrada'
db = wrds.Connection(wrds_username=WRDS_USERNAME)

## Configuration

In [ ]:
START_DATE = '2020-01-01'
END_DATE   = '2025-12-31'  # CIZ extends here; legacy capped at 2024-12-31

OUT_DIR = Path('../data/wrds')
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1) Trading days from CRSP DSF_V2 (CIZ)

Distinct `dlycaldt`. Legacy used `crsp.dsi` which caps at 2024-12-31.

In [ ]:
trading_days = db.raw_sql(f"""
    SELECT DISTINCT dlycaldt AS date
    FROM crsp.dsf_v2
    WHERE dlycaldt BETWEEN '{START_DATE}' AND '{END_DATE}'
    ORDER BY dlycaldt
""", date_cols=['date'])

trading_days['date'] = pd.to_datetime(trading_days['date']).dt.normalize()

trading_days.to_parquet(OUT_DIR / 'trading_days.parquet')
print(f"Saved -> {OUT_DIR / 'trading_days.parquet'}")
print(f"Trading days: {len(trading_days):,}")
print(f"Span: {trading_days['date'].min()} -> {trading_days['date'].max()}")

## 2) S&P 500 daily membership (DSP500LIST_V2)

Same table in CIZ. Trading-days source switched to `dsf_v2`.

In [ ]:
sp500_membership = db.raw_sql(f"""
    SELECT t.date,
           m.permno,
           m.mbrstartdt AS mbr_startdt,
           m.mbrenddt   AS mbr_enddt
    FROM (SELECT DISTINCT dlycaldt AS date FROM crsp.dsf_v2
          WHERE dlycaldt BETWEEN '{START_DATE}' AND '{END_DATE}') t
    INNER JOIN crsp.dsp500list_v2 m
      ON t.date BETWEEN m.mbrstartdt AND COALESCE(m.mbrenddt, '9999-12-31'::date)
    ORDER BY t.date, m.permno
""", date_cols=['date', 'mbr_startdt', 'mbr_enddt'])

print(f"Membership rows: {len(sp500_membership):,}")
sp500_membership.head()

## 3) Stock names valid on day J (STOCKNAMES_V2) -> constituents

CIZ filter for common stock: `securitytype='EQTY' AND securitysubtype='COM' AND sharetype IN ('NS', 'AD', 'SB', 'UG')`.

Aliases applied:
- `issuernm` -> `comnam`
- `shareclass` -> `shrcls`
- `usincflg` -> `shrcd` (Y -> 10, N -> 11)

In [ ]:
sp500_day_raw = db.raw_sql(f"""
    SELECT m.date,
           m.permno,
           m.mbrstartdt AS mbr_startdt,
           m.mbrenddt   AS mbr_enddt,
           s.permco,
           s.ticker,
           s.issuernm   AS comnam,
           s.cusip,
           CASE WHEN s.usincflg = 'Y' THEN 10
                WHEN s.usincflg = 'N' THEN 11
                ELSE NULL END AS shrcd,
           s.shareclass AS shrcls,
           s.namedt,
           COALESCE(s.nameenddt, '9999-12-31'::date) AS nameenddt
    FROM (
        SELECT t.date, m.permno, m.mbrstartdt, m.mbrenddt
        FROM (SELECT DISTINCT dlycaldt AS date FROM crsp.dsf_v2
              WHERE dlycaldt BETWEEN '{START_DATE}' AND '{END_DATE}') t
        INNER JOIN crsp.dsp500list_v2 m
          ON t.date BETWEEN m.mbrstartdt AND COALESCE(m.mbrenddt, '9999-12-31'::date)
    ) m
    LEFT JOIN crsp.stocknames_v2 s
      ON  m.permno = s.permno
      AND m.date BETWEEN s.namedt AND COALESCE(s.nameenddt, '9999-12-31'::date)
      AND s.securitytype    = 'EQTY'
      AND s.securitysubtype = 'COM'
      AND s.sharetype IN ('NS', 'AD', 'SB', 'UG')
""", date_cols=['date', 'mbr_startdt', 'mbr_enddt', 'namedt', 'nameenddt'])

# Dedupe: 1 row per (date, permno), keeping the most recent namedt
sp500_day = (sp500_day_raw
             .sort_values(['date', 'permno', 'namedt', 'nameenddt'],
                          ascending=[True, True, False, False])
             .drop_duplicates(subset=['date', 'permno'], keep='first')
             .reset_index(drop=True))

print(f"Constituents daily rows: {len(sp500_day):,}")
sp500_day.head()

In [ ]:
# Persist constituents (no GVKEY)
sp500_constituents_daily = sp500_day.assign(
    ticker = sp500_day['ticker'].str.upper(),
    comnam = sp500_day['comnam'].str.upper(),
    cusip  = sp500_day['cusip'].str.upper(),
).sort_values(['date', 'permno']).reset_index(drop=True)

sp500_constituents_daily.to_parquet(OUT_DIR / 'sp500_constituents_daily.parquet')
print(f"Saved -> {OUT_DIR / 'sp500_constituents_daily.parquet'}")
print(f"Rows: {len(sp500_constituents_daily):,}")
print(f"Unique PERMNO: {sp500_constituents_daily['permno'].nunique():,}")
print(f"Unique days:   {sp500_constituents_daily['date'].nunique():,}")

## 4) Daily prices / returns / liquidity (DSF_V2)

Restrict to S&P 500 universe, chunked by year. All CIZ columns aliased to legacy names so downstream cells (panel build, monthly aggregation) work unchanged.

In [ ]:
univ_permno = tuple(int(p) for p in sp500_constituents_daily['permno'].unique())
print(f"Universe PERMNO count: {len(univ_permno):,}")

years = range(int(START_DATE[:4]), int(END_DATE[:4]) + 1)

dsf_chunks = []
for y in years:
    y_start = max(f'{y}-01-01', START_DATE)
    y_end   = min(f'{y}-12-31', END_DATE)

    chunk = db.raw_sql(f"""
        SELECT dlycaldt     AS date,
               permno,
               permco,
               dlyprc       AS prc,
               dlyretx      AS retx,
               dlyret       AS ret,
               dlyvol       AS vol,
               shrout,
               dlylow       AS bidlo,
               dlyhigh      AS askhi,
               dlycumfacpr  AS cfacpr,
               dlycumfacshr AS cfacshr,
               primaryexch  AS hexcd,
               siccd        AS hsiccd,
               nasdissuno   AS issuno,
               cusip        AS cusip_hdr,
               dlybid       AS bid,
               dlyask       AS ask,
               dlyopen      AS openprc,
               dlynumtrd    AS numtrd
        FROM crsp.dsf_v2
        WHERE dlycaldt BETWEEN '{y_start}' AND '{y_end}'
          AND permno IN {univ_permno}
    """, date_cols=['date'])
    dsf_chunks.append(chunk)
    print(f"  {y}: {len(chunk):,} rows")

dsf_core = pd.concat(dsf_chunks, ignore_index=True)
del dsf_chunks
print(f"Total DSF rows: {len(dsf_core):,}")

## 5) Distributions & splits (STKDISTRIBUTIONS)

Same daily aggregation logic as the legacy notebook. CIZ aliases are applied in the SELECT so the pandas groupby code below uses identical column names.

In [ ]:
dist_raw = db.raw_sql(f"""
    SELECT permno,
           disexdt        AS date,
           disdivamt      AS divamt,
           disfacpr       AS facpr,
           disfacshr      AS facshr,
           disdetailtype  AS distcd,
           dispaydt       AS paydt,
           disrecorddt    AS rcrddt
    FROM crsp.stkdistributions
    WHERE disexdt BETWEEN '{START_DATE}' AND '{END_DATE}'
""", date_cols=['date', 'paydt', 'rcrddt'])

# Compute log-products of facpr / facshr defensively (handle <=0 and missing)
dist_daily_agg = (dist_raw
    .assign(
        log_facpr_pos  = np.where(dist_raw['facpr']  > 0, np.log(dist_raw['facpr'].clip(lower=1e-30)),  0.0),
        log_facshr_pos = np.where(dist_raw['facshr'] > 0, np.log(dist_raw['facshr'].clip(lower=1e-30)), 0.0),
        divamt_filled  = dist_raw['divamt'].fillna(0.0),
    )
    .groupby(['permno', 'date'], as_index=False)
    .agg(
        divamt_sum       = ('divamt_filled', 'sum'),
        facpr_log_sum    = ('log_facpr_pos', 'sum'),
        facshr_log_sum   = ('log_facshr_pos', 'sum'),
        dist_event_count = ('distcd', 'size'),
        distcd_min       = ('distcd', 'min'),
        distcd_max       = ('distcd', 'max'),
        paydt_min        = ('paydt', 'min'),
        paydt_max        = ('paydt', 'max'),
        rcrddt_min       = ('rcrddt', 'min'),
        rcrddt_max       = ('rcrddt', 'max'),
    )
    .assign(
        facpr_prod  = lambda d: np.exp(d['facpr_log_sum']),
        facshr_prod = lambda d: np.exp(d['facshr_log_sum']),
    )
    .drop(columns=['facpr_log_sum', 'facshr_log_sum']))

print(f"Distribution events aggregated: {len(dist_daily_agg):,}")
dist_daily_agg.head()

## 6) Delistings (STKDELISTS)

In [ ]:
delist_daily = db.raw_sql(f"""
    SELECT permno,
           delistingdt    AS date,
           delreasontype  AS dlstcd,
           deldtprc       AS dlprc,
           delret         AS dlret
    FROM crsp.stkdelists
    WHERE delistingdt BETWEEN '{START_DATE}' AND '{END_DATE}'
    ORDER BY delistingdt, permno
""", date_cols=['date'])

print(f"Delistings: {len(delist_daily):,}")

## 7) Market index (CRSP DSI - LEGACY)

**Note**: there is no `dsi_v2` in CIZ. We keep the legacy `crsp.dsi` here. Rows for 2025+ will get NULL for VWRETD/EWRETD/SPRTRN/etc. — graceful degradation.

In [ ]:
dsi_core = db.raw_sql(f"""
    SELECT date,
           vwretd, ewretd,
           vwretx, ewretx,
           sprtrn, spindx,
           totcnt, usdcnt, totval, usdval
    FROM crsp.dsi
    WHERE date BETWEEN '{START_DATE}' AND '{END_DATE}'
""", date_cols=['date'])

print(f"DSI rows: {len(dsi_core):,}")
print(f"Span: {dsi_core['date'].min()} -> {dsi_core['date'].max()}")

## 8) Build daily panel `sp500_daily_full`

Identical to the legacy notebook — CIZ aliases mean the merged DataFrame has the same column names.

In [ ]:
univ = sp500_constituents_daily[[
    'date', 'permno', 'permco', 'mbr_startdt', 'mbr_enddt',
    'ticker', 'comnam', 'cusip', 'shrcd', 'shrcls', 'namedt', 'nameenddt'
]]

dsf_for_join = dsf_core.drop(columns=['permco'])

panel = (univ
    .merge(dsf_for_join,    on=['date', 'permno'], how='left')
    .merge(dist_daily_agg,  on=['date', 'permno'], how='left')
    .merge(delist_daily,    on=['date', 'permno'], how='left')
    .merge(dsi_core,        on='date',             how='left')
)

# WRDS may return nullable Int64/Float64 dtypes -> force float64 to avoid
# 'boolean value of NA is ambiguous' in np.where below.
numeric_cols = [
    'prc', 'retx', 'ret', 'vol', 'shrout', 'bidlo', 'askhi', 'cfacpr', 'cfacshr',
    'hsiccd', 'issuno', 'bid', 'ask', 'openprc', 'numtrd',
    'divamt_sum', 'facpr_prod', 'facshr_prod', 'dist_event_count',
    'dlprc', 'dlret',
    'vwretd', 'ewretd', 'vwretx', 'ewretx', 'sprtrn', 'spindx',
    'totcnt', 'usdcnt', 'totval', 'usdval',
]
for col in numeric_cols:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors='coerce').astype('float64')

# Derivatives
panel['me']         = panel['prc'].abs() * panel['shrout']
panel['turnover']   = np.where(panel['shrout'] > 0, panel['vol'] / panel['shrout'], np.nan)
panel['dollar_vol'] = panel['prc'].abs() * panel['vol']

# RET_COMBINED: splice delisting return into total return
panel['ret_combined'] = np.where(
    panel['dlret'].notna(),
    (1 + panel['ret'].fillna(0)) * (1 + panel['dlret']) - 1,
    panel['ret']
)

sp500_daily_full = panel.sort_values(['date', 'permno']).reset_index(drop=True)
sp500_daily_full.to_parquet(OUT_DIR / 'sp500_daily_full.parquet')
print(f"Saved -> {OUT_DIR / 'sp500_daily_full.parquet'}")
print(f"Rows: {len(sp500_daily_full):,}, Cols: {sp500_daily_full.shape[1]}")

### Sanity checks (daily)

In [ ]:
checks = pd.Series({
    'n_rows':              len(sp500_daily_full),
    'n_unique_keys':       sp500_daily_full[['date', 'permno']].drop_duplicates().shape[0],
    'n_ret':               sp500_daily_full['ret'].notna().sum(),
    'n_dlret':             sp500_daily_full['dlret'].notna().sum(),
    'n_ret_combined':      sp500_daily_full['ret_combined'].notna().sum(),
    'n_missing_price_days': ((sp500_daily_full['prc'].isna()) & (sp500_daily_full['ret'].isna())).sum(),
    'n_div_days':          (sp500_daily_full['divamt_sum'].fillna(0) > 0).sum(),
    'n_vwretx_filled':     sp500_daily_full['vwretx'].notna().sum(),  # caps at 2024-12-31
})
checks

## 9) Monthly version `sp500_monthly_full`

Same rules as the legacy notebook:
- `date` = last trading day of the month (per PERMNO).
- Levels (PRC, SHROUT, BID, ASK, ...) = last day's value.
- Flows (VOL, DOLLAR_VOL, NUMTRD, DIVAMT_SUM, ...) = monthly sum.
- Returns (RET, RETX, RET_COMBINED, VWRETD, ...) = compounded via `exp(sum(log(1+r))) - 1`.

In [ ]:
df = sp500_daily_full.copy()
df['mdate'] = (df['date'] + pd.offsets.MonthEnd(0)).dt.normalize()

# 1) Last observed day in month per PERMNO
last_in_month = (df.groupby(['permno', 'mdate'], as_index=False)['date']
                   .max()
                   .rename(columns={'date': 'last_dt'}))

# 2) Levels at last day of month
level_cols = [
    'permco', 'mbr_startdt', 'mbr_enddt',
    'ticker', 'comnam', 'cusip', 'shrcd', 'shrcls', 'namedt', 'nameenddt',
    'prc', 'shrout', 'bidlo', 'askhi', 'openprc',
    'hexcd', 'hsiccd', 'issuno', 'cusip_hdr', 'bid', 'ask',
    'dlstcd', 'dlprc', 'dlret', 'spindx',
]
levels_last = (df.merge(last_in_month, on=['permno', 'mdate'])
                 .query('date == last_dt')
                 [['permno', 'mdate', 'last_dt'] + level_cols])

# 3) Per-PERMNO monthly aggregation (sums + compounded returns)
def compound(s: pd.Series) -> float:
    s = s.dropna()
    return np.expm1(np.log1p(s).sum()) if len(s) else np.nan

agg_month = (df.groupby(['permno', 'mdate']).agg(
    vol              = ('vol',              'sum'),
    dollar_vol       = ('dollar_vol',       'sum'),
    numtrd           = ('numtrd',           'sum'),
    divamt_sum       = ('divamt_sum',       'sum'),
    dist_event_count = ('dist_event_count', 'sum'),
    distcd_min       = ('distcd_min',       'min'),
    distcd_max       = ('distcd_max',       'max'),
    paydt_min        = ('paydt_min',        'min'),
    paydt_max        = ('paydt_max',        'max'),
    rcrddt_min       = ('rcrddt_min',       'min'),
    rcrddt_max       = ('rcrddt_max',       'max'),
    n_trd_days       = ('prc', lambda s: ((s.notna()) | (df.loc[s.index, 'ret'].notna())).sum()),
    ret              = ('ret',              compound),
    retx             = ('retx',             compound),
    ret_combined     = ('ret_combined',     compound),
).reset_index())

# 4) Monthly benchmarks (one set of values per mdate)
market_month = (df.groupby('mdate').agg(
    vwretd = ('vwretd', compound),
    ewretd = ('ewretd', compound),
    vwretx = ('vwretx', compound),
    ewretx = ('ewretx', compound),
    sprtrn = ('sprtrn', compound),
    totcnt = ('totcnt', 'sum'),
    usdcnt = ('usdcnt', 'sum'),
    totval = ('totval', 'sum'),
    usdval = ('usdval', 'sum'),
).reset_index())

# 5) Final assembly
sp500_monthly_full = (levels_last
    .merge(agg_month,    on=['permno', 'mdate'], how='left')
    .merge(market_month, on='mdate',             how='left')
    .rename(columns={'mdate': 'date'})
    .sort_values(['date', 'permno'])
    .reset_index(drop=True))

sp500_monthly_full.to_parquet(OUT_DIR / 'sp500_monthly_full.parquet')
print(f"Saved -> {OUT_DIR / 'sp500_monthly_full.parquet'}")
print(f"Rows: {len(sp500_monthly_full):,}, Months: {sp500_monthly_full['date'].nunique():,}")

### Sanity checks (monthly)

In [ ]:
monthly_checks = pd.Series({
    'n_rows':         len(sp500_monthly_full),
    'n_months':       sp500_monthly_full['date'].nunique(),
    'n_permno':       sp500_monthly_full['permno'].nunique(),
    'n_unique_keys':  sp500_monthly_full[['date', 'permno']].drop_duplicates().shape[0],
})
monthly_checks

## 10) Disconnect

In [ ]:
db.close()